In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, concatenate, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import pickle

# Load dataset
df = pd.read_csv('WELFake_Dataset.csv')

# Drop rows where 'title' or 'text' is missing
df = df.dropna(subset=['title', 'text'])

# Combine title and text for maximum context
df['full_text'] = df['title'].astype(str) + " " + df['text'].astype(str)

# Ensure label is integers (1 = Real, 0 = Fake usually, but check dataset specifics)
X = df['full_text'].values
y = df['label'].values

print(f"Total articles loaded: {len(X)}")

Total articles loaded: 71537


In [ ]:
# 1. Text Preprocessing for TF-IDF (Branch A)
max_tfidf_features = 10000
tfidf_vectorizer = TfidfVectorizer(max_features=max_tfidf_features, stop_words='english')
X_tfidf = tfidf_vectorizer.fit_transform(X).toarray() # Convert sparse matrix to dense array

# 2. Text Preprocessing for LSTM (Branch B)
max_sequence_length = 500
max_words = 10000 # Vocabulary size for LSTM embedding

In [ ]:
print("Pre-truncating text to 500 words to save Keras CPU time...")
# Split each article by spaces, grab the first 500 words, and rejoin them.
df['short_text'] = df['full_text'].apply(lambda x: " ".join(str(x).split()[:500]))
X_short = df['short_text'].values

print("Starting LSTM Tokenization...")
max_sequence_length = 500
max_words = 10000

tokenizer = Tokenizer(num_words=max_words)
# We now fit the tokenizer ONLY on the shortened 500-word strings
tokenizer.fit_on_texts(X_short)
X_seq = tokenizer.texts_to_sequences(X_short)
X_seq_padded = pad_sequences(X_seq, maxlen=max_sequence_length)

print(f"Padded Sequences created successfully. Shape: {X_seq_padded.shape}")

Pre-truncating text to 500 words to save Keras CPU time...
Starting LSTM Tokenization...
Padded Sequences created successfully. Shape: (71537, 500)


In [ ]:
X_train_tfidf, X_test_tfidf, X_train_seq, X_test_seq, y_train, y_test = train_test_split(
    X_tfidf, X_seq_padded, y, test_size=0.2, random_state=42
)

In [ ]:
# --- Branch A: TF-IDF + Dense NN ---
input_tfidf = Input(shape=(max_tfidf_features,), name="tfidf_input")
dense_1 = Dense(128, activation='relu')(input_tfidf)
dropout_1 = Dropout(0.5)(dense_1)
dense_2 = Dense(64, activation='relu')(dropout_1)

# --- Branch B: Token Sequence + LSTM ---
input_seq = Input(shape=(max_sequence_length,), name="seq_input")
embedding = Embedding(input_dim=max_words, output_dim=128, input_length=max_sequence_length)(input_seq)
lstm_layer = LSTM(64)(embedding)

# --- Fusion Layer ---
merged = concatenate([dense_2, lstm_layer])
merged_dense = Dense(32, activation='relu')(merged)
merged_dropout = Dropout(0.5)(merged_dense)

# Output Layer (Binary Classification)
output_layer = Dense(1, activation='sigmoid', name="output")(merged_dropout)

# Define and Compile Model
model = Model(inputs=[input_tfidf, input_seq], outputs=output_layer)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

In [ ]:
# Save the Keras model
model.save('dual_branch_fakenews_model.h5')

# Save the preprocessors using pickle
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Model and preprocessors saved successfully!")

In [ ]:
import pandas as pd
import numpy as np
import gc
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, concatenate, Dropout

print("1. Loading and Preprocessing Data...")
# Load dataset
df = pd.read_csv('WELFake_Dataset.csv')
df = df.dropna(subset=['title', 'text'])

# Combine title and text first
df['full_text'] = df['title'].astype(str) + " " + df['text'].astype(str)

# Split by spaces, grab the first 500 words, and rejoin them safely
df['short_text'] = df['full_text'].apply(lambda x: " ".join(x.split()[:500]))

X_raw = df['short_text'].values
y = df['label'].values

print("2. Splitting Data...")
# Split raw text FIRST to avoid duplicating massive TF-IDF arrays in memory
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

# Free up memory immediately
del df
del X_raw
gc.collect()

print("3. Building TF-IDF Features (Branch A)...")
max_tfidf_features = 5000
# Force float32 to cut memory consumption in half
tfidf_vectorizer = TfidfVectorizer(max_features=max_tfidf_features, stop_words='english', dtype=np.float32)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text).toarray()
X_test_tfidf = tfidf_vectorizer.transform(X_test_text).toarray()
gc.collect()

print("4. Building Sequence Features (Branch B)...")
max_sequence_length = 500
max_words = 10000

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train_text) # Fit ONLY on training data

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=max_sequence_length)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=max_sequence_length)

# Final memory cleanup before training
del X_train_text
del X_test_text
gc.collect()

print("5. Building Dual-Branch Keras Model...")
# --- Branch A: TF-IDF + Dense NN ---
input_tfidf = Input(shape=(max_tfidf_features,), name="tfidf_input")
dense_1 = Dense(128, activation='relu')(input_tfidf)
dropout_1 = Dropout(0.5)(dense_1)
dense_2 = Dense(64, activation='relu')(dropout_1)

# --- Branch B: Token Sequence + LSTM ---
input_seq = Input(shape=(max_sequence_length,), name="seq_input")
embedding = Embedding(input_dim=max_words, output_dim=128, input_length=max_sequence_length)(input_seq)
lstm_layer = LSTM(64)(embedding)

# --- Fusion Layer ---
merged = concatenate([dense_2, lstm_layer])
merged_dense = Dense(32, activation='relu')(merged)
merged_dropout = Dropout(0.5)(merged_dense)

# Output Layer
output_layer = Dense(1, activation='sigmoid', name="output")(merged_dropout)

model = Model(inputs=[input_tfidf, input_seq], outputs=output_layer)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("6. Training Model...")
# Use batch_size=32 to prevent GPU Out-Of-Memory errors
history = model.fit(
    {"tfidf_input": X_train_tfidf, "seq_input": X_train_seq},
    y_train,
    validation_data=({"tfidf_input": X_test_tfidf, "seq_input": X_test_seq}, y_test),
    epochs=5,
    batch_size=32
)

print("7. Saving Model and Preprocessors...")
# Save everything needed for the FastAPI backend
model.save('dual_branch_fakenews_model.h5')

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Pipeline Complete! Download the .h5 and .pkl files from the Colab file explorer.")

1. Loading and Preprocessing Data...
2. Splitting Data...
3. Building TF-IDF Features (Branch A)...
4. Building Sequence Features (Branch B)...
5. Building Dual-Branch Keras Model...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


6. Training Model...
Epoch 1/5
1789/1789 ━━━━━━━━━━━━━━━━━━━━ 690s 384ms/step - accuracy: 0.9332 - loss: 0.1766 - val_accuracy: 0.9600 - val_loss: 0.1087
Epoch 2/5
1789/1789 ━━━━━━━━━━━━━━━━━━━━ 663s 371ms/step - accuracy: 0.9684 - loss: 0.0906 - val_accuracy: 0.9599 - val_loss: 0.1143
Epoch 3/5
1789/1789 ━━━━━━━━━━━━━━━━━━━━ 688s 374ms/step - accuracy: 0.9855 - loss: 0.0448 - val_accuracy: 0.9690 - val_loss: 0.1070
Epoch 4/5
1789/1789 ━━━━━━━━━━━━━━━━━━━━ 670s 374ms/step - accuracy: 0.9934 - loss: 0.0207 - val_accuracy: 0.9688 - val_loss: 0.1376
Epoch 5/5
1789/1789 ━━━━━━━━━━━━━━━━━━━━ 669s 374ms/step - accuracy: 0.9960 - loss: 0.0123 - val_accuracy: 0.9685 - val_loss: 0.1533


7. Saving Model and Preprocessors...
Pipeline Complete! Download the .h5 and .pkl files from the Colab file explorer.


In [ ]:
!zip fakenews_project_files.zip dual_branch_fakenews_model.h5 tfidf_vectorizer.pkl tokenizer.pkl

# 2. Force the browser to download the zip archive
from google.colab import files
files.download('fakenews_project_files.zip')

  adding: dual_branch_fakenews_model.h5 (deflated 6%)
  adding: tfidf_vectorizer.pkl (deflated 66%)
  adding: tokenizer.pkl (deflated 58%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# New Section